# Trabalho Prático 1 — Gerador de Horário Escolar (CP-SAT)

Este notebook contém:
- justificação da modelação e das ferramentas,
- leitura dos dados a partir de ficheiros CSV (nada escrito no código),
- modelo CP-SAT (OR-Tools) com os requisitos R1–R7 e o objetivo O1 (buracos),
- validador independente do solver (R1–R7) e testes,
- construção incremental (R9): reparação local por vizinhanças + *hints*,
- comparação `H1` incremental vs `H1` do zero, outros cenários, dataset de teste e escala.

**Instalação:**

In [ ]:
pip install ortools pandas


### Decisões de modelação (justificação)

- **Técnica:** CP-SAT do OR-Tools. O problema é discreto, com muitas restrições de cardinalidade ("no máximo 1", "exatamente carga") e um objetivo linear (buracos); o CP-SAT trata isto bem e permite *hints*, fixar variáveis e limites de tempo, que são essenciais para o R9.
- **Variáveis:** para cada `(turma, disciplina, dia, período)` uma booleana `s` que indica que **começa** uma aula/bloco nesse tempo. Em disciplinas normais o bloco tem 1 tempo; em `duplo_periodo=sim` ocupa o tempo `p` e `p+1` (por isso `p` só vai até ao penúltimo tempo — R4 fica garantido por construção).
- **R2:** soma das variáveis = `carga` (ou `carga/2` blocos nas duplas). **R3:** soma por dia ≤ 1. **R1:** soma das ocupações da turma por tempo ≤ 1.
- **R5/R6:** ocupação do professor por tempo ≤ 1 e = 0 nos tempos indisponíveis.
- **R7:** as salas são tratadas como *pools* por tipo (`normal` agrega todas as normais; as especiais são identificadas pelo nome). Em cada tempo, nº de aulas do pool ≤ `quantidade`. A sala concreta ("Sala Normal 3") é atribuída *a posteriori* (guloso), porque dentro de um pool as salas são equivalentes.
- **O1 (buracos):** por professor/dia, variáveis `antes[p]` e `depois[p]` (existe aula em `≤p` / `≥p`) e `buraco[p] ≥ antes[p] + depois[p] − trabalha[p] − 1`.
- **R9 (incremental):** (1) deteta as aulas de `H0` que deixam de ser válidas com os novos dados; (2) liberta só as `(turma, disciplina)` afetadas e **fixa** todas as outras ao valor de `H0`; (3) se for infeasible, alarga a vizinhança (professores afetados → turmas afetadas); (4) em último caso resolve o modelo completo com `H0` como *hint*. Em todas as fases a função objetivo é lexicográfica: primeiro **minimizar aulas alteradas** em relação a `H0`, depois buracos.
- **Dados:** `pandas`, para ler os CSV com `dtype=str` e validar os valores.

In [ ]:
## 1. Importações e funções utilitárias
import os
import copy
import time
import tempfile
from collections import defaultdict

import pandas as pd
from ortools.sat.python import cp_model
from IPython.display import display

DIAS = ["Seg", "Ter", "Qua", "Qui", "Sex"]   # parâmetros da semana (não são dados de entrada)
N_PERIODOS = 5
POOL_NORMAL = "__normal__"

## 2. Leitura dos dados (R8)

Todos os dados vêm dos CSV. A função devolve uma estrutura em memória:
- `turmas`: lista de turmas,
- `disc`: `{disciplina: {prof, carga, duplo, pool}}`,
- `salas`: `{pool: quantidade}`,
- `indisp`: conjunto `(professor, dia_idx, periodo_idx)` de indisponibilidades.

In [ ]:
def _ler(pasta, nome):
    df = pd.read_csv(os.path.join(pasta, nome), dtype=str, keep_default_na=False)
    df.columns = [c.strip() for c in df.columns]
    return df.apply(lambda col: col.str.strip())


def carregar_dados(pasta):
    turmas_df = _ler(pasta, "turmas.csv")
    disc_df = _ler(pasta, "disciplinas.csv")
    salas_df = _ler(pasta, "salas.csv")
    disp_df = _ler(pasta, "disponibilidade_excecoes.csv")

    turmas = list(turmas_df["turma"])

    salas = {}
    for _, r in salas_df.iterrows():
        pool = POOL_NORMAL if r["tipo"].lower() == "normal" else r["sala"]
        salas[pool] = salas.get(pool, 0) + int(r["quantidade"])

    disc = {}
    for _, r in disc_df.iterrows():
        pool = r["sala_especial"] if r["sala_especial"] else POOL_NORMAL
        if pool not in salas:
            raise ValueError(f"Sala '{pool}' da disciplina {r['disciplina']} não existe em salas.csv")
        carga = int(r["carga_semanal"])
        duplo = r["duplo_periodo"].lower() == "sim"
        if duplo and carga % 2 != 0:
            raise ValueError(f"{r['disciplina']}: duplo_periodo exige carga par")
        disc[r["disciplina"]] = {"prof": r["professor"], "carga": carga, "duplo": duplo, "pool": pool}

    indisp = {(r["professor"], DIAS.index(r["dia"]), int(r["periodo"]) - 1)
              for _, r in disp_df.iterrows()}

    return {"turmas": turmas, "disc": disc, "salas": salas, "indisp": indisp}

## 3. Representação do horário e métricas

Um horário é guardado como o conjunto `S` de **inícios de bloco** `(turma, disciplina, dia, periodo)`.
`blocos_para_aulas` expande `S` para aulas por tempo (um bloco duplo dá 2 aulas).

In [ ]:
def blocos_para_aulas(d, S):
    aulas = []
    for (c, k, dia, p) in sorted(S):
        aulas.append((c, k, dia, p))
        if d["disc"][k]["duplo"]:
            aulas.append((c, k, dia, p + 1))
    return aulas


def atribuir_salas(d, aulas):
    """Atribui a sala concreta dentro de cada pool (guloso, por tempo)."""
    grupos = defaultdict(list)
    for a in aulas:
        c, k, dia, p = a
        grupos[(d["disc"][k]["pool"], dia, p)].append(a)
    res = {}
    for (pool, dia, p), lst in grupos.items():
        nome = "Sala Normal" if pool == POOL_NORMAL else pool
        for i, a in enumerate(sorted(lst)):
            res[a] = f"{nome} {i + 1}"
    return res


def contar_buracos(d, aulas):
    por = defaultdict(set)
    for c, k, dia, p in aulas:
        por[(d["disc"][k]["prof"], dia)].add(p)
    return sum((max(ps) - min(ps) + 1) - len(ps) for ps in por.values())


def aulas_alteradas(S0, S1):
    """Nº de blocos de S1 que não existiam em S0 (só turmas já presentes em H0)."""
    turmas_comuns = {c for c, _, _, _ in S0}
    return len({key for key in (S1 - S0) if key[0] in turmas_comuns})


def mostrar_horario(d, S, turma):
    aulas = blocos_para_aulas(d, S)
    salas = atribuir_salas(d, aulas)
    grid = pd.DataFrame("", index=[f"T{p + 1}" for p in range(N_PERIODOS)], columns=DIAS)
    for a, sala in salas.items():
        c, k, dia, p = a
        if c == turma:
            grid.iloc[p, dia] = f"{k} ({d['disc'][k]['prof']}) [{sala}]"
    return grid

## 4. Modelo CP-SAT

`construir_modelo(d, ref, livres)`:
- `ref` (opcional): horário `H0` (conjunto de inícios de bloco). Se existir, usa-se como *hint* e a função objetivo passa a ser lexicográfica (aulas alteradas ≫ buracos).
- `livres` (opcional): conjunto de pares `(turma, disciplina)` que podem mudar. Os pares de `H0` que **não** estejam em `livres` ficam fixos.

In [ ]:
def construir_modelo(d, ref=None, livres=None):
    m = cp_model.CpModel()
    D, P = len(DIAS), N_PERIODOS
    turmas, disc = d["turmas"], d["disc"]

    var = {}                        # (turma, disc, dia, p) -> BoolVar (início de bloco)
    ocupa = defaultdict(list)       # (turma, disc, dia, p) -> vars que ocupam esse tempo

    for c in turmas:
        for k, info in disc.items():
            dup = info["duplo"]
            n_pos = P - 1 if dup else P                      # R4: bloco duplo cabe no dia
            for dia in range(D):
                for p in range(n_pos):
                    v = m.NewBoolVar(f"s_{c}_{k}_{dia}_{p}")
                    var[(c, k, dia, p)] = v
                    ocupa[(c, k, dia, p)].append(v)
                    if dup:
                        ocupa[(c, k, dia, p + 1)].append(v)
                m.Add(sum(var[(c, k, dia, p)] for p in range(n_pos)) <= 1)        # R3
            n_blocos = info["carga"] // 2 if dup else info["carga"]
            m.Add(sum(var[(c, k, dia, p)] for dia in range(D) for p in range(n_pos)) == n_blocos)  # R2

    # R1: uma turma não tem duas aulas em simultâneo
    for c in turmas:
        for dia in range(D):
            for p in range(P):
                m.Add(sum(v for k in disc for v in ocupa.get((c, k, dia, p), [])) <= 1)

    # R5 + R6: professores
    profs = sorted({i["prof"] for i in disc.values()})
    trabalha = {}
    for pr in profs:
        ks = [k for k, i in disc.items() if i["prof"] == pr]
        for dia in range(D):
            for p in range(P):
                expr = sum(v for c in turmas for k in ks for v in ocupa.get((c, k, dia, p), []))
                w = m.NewBoolVar(f"w_{pr}_{dia}_{p}")
                m.Add(w == expr)                              # R5 (expr ∈ {0,1})
                if (pr, dia, p) in d["indisp"]:
                    m.Add(w == 0)                             # R6
                trabalha[(pr, dia, p)] = w

    # R7: capacidade das salas por pool
    for pool, qtd in d["salas"].items():
        ks = [k for k, i in disc.items() if i["pool"] == pool]
        for dia in range(D):
            for p in range(P):
                m.Add(sum(v for c in turmas for k in ks for v in ocupa.get((c, k, dia, p), [])) <= qtd)

    # O1: buracos por professor/dia
    buracos = []
    for pr in profs:
        for dia in range(D):
            antes = [m.NewBoolVar(f"a_{pr}_{dia}_{p}") for p in range(P)]
            depois = [m.NewBoolVar(f"d_{pr}_{dia}_{p}") for p in range(P)]
            for p in range(P):
                w = trabalha[(pr, dia, p)]
                m.Add(antes[p] >= w)
                m.Add(depois[p] >= w)
                if p > 0:
                    m.Add(antes[p] >= antes[p - 1])
                if p < P - 1:
                    m.Add(depois[p] >= depois[p + 1])
                b = m.NewBoolVar(f"b_{pr}_{dia}_{p}")
                m.Add(b >= antes[p] + depois[p] - w - 1)
                buracos.append(b)
    obj_buracos = sum(buracos)

    # Objetivo (+ hint e fixações para o modo incremental)
    if ref is not None:
        perdidos = [1 - var[key] for key in ref if key in var]   # blocos de H0 que deixam de existir
        peso = len(buracos) + 1                                  # lexicográfico: mudanças ≫ buracos
        m.Minimize(peso * sum(perdidos) + obj_buracos)
        for key, v in var.items():
            m.AddHint(v, 1 if key in ref else 0)
        if livres is not None:
            pares_ref = {(c, k) for (c, k, _, _) in ref}
            for (c, k, dia, p), v in var.items():
                if (c, k) in pares_ref and (c, k) not in livres:
                    m.Add(v == (1 if (c, k, dia, p) in ref else 0))
    else:
        m.Minimize(obj_buracos)

    return m, var, buracos

## 5. Resolução

`resolver` constrói e resolve o modelo, medindo o tempo total (construção + solver) e o tempo até à 1ª solução.

In [ ]:
class _PrimeiraSolucao(cp_model.CpSolverSolutionCallback):
    def __init__(self):
        super().__init__()
        self.t = None

    def on_solution_callback(self):
        if self.t is None:
            self.t = time.perf_counter()


def resolver(d, ref=None, livres=None, tempo=20.0, workers=8):
    t_ini = time.perf_counter()
    m, var, _ = construir_modelo(d, ref, livres)
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = tempo
    solver.parameters.num_workers = workers
    cb = _PrimeiraSolucao()
    st = solver.Solve(m, cb)
    dt = time.perf_counter() - t_ini
    ok = st in (cp_model.OPTIMAL, cp_model.FEASIBLE)
    S = {key for key, v in var.items() if solver.Value(v)} if ok else None
    return {
        "estado": solver.StatusName(st),
        "S": S,
        "tempo": dt,
        "t_primeira": (cb.t - t_ini) if cb.t is not None else None,
        "fase": "—",
    }

## 6. Validação independente (R1–R7)

O validador **não usa o modelo**: recebe só a lista de aulas por tempo e verifica cada requisito. Devolve uma lista de `(código, mensagem)`.

In [ ]:
def validar(d, aulas):
    L = list(aulas)
    erros = []

    # R1
    cont = defaultdict(int)
    for c, k, dia, p in L:
        cont[(c, dia, p)] += 1
    erros += [("R1", f"Turma {c} com {n} aulas em {DIAS[dia]} T{p + 1}")
              for (c, dia, p), n in cont.items() if n > 1]

    # R2, R3, R4
    g = defaultdict(lambda: defaultdict(list))
    for c, k, dia, p in L:
        g[(c, k)][dia].append(p)
    for c in d["turmas"]:
        for k, info in d["disc"].items():
            por_dia = g.get((c, k), {})
            total = sum(len(ps) for ps in por_dia.values())
            if total != info["carga"]:
                erros.append(("R2", f"{c}/{k}: {total} tempos em vez de {info['carga']}"))
            for dia, ps in por_dia.items():
                ps = sorted(ps)
                if info["duplo"]:
                    if len(ps) > 2:
                        erros.append(("R3", f"{c}/{k}: mais de um bloco em {DIAS[dia]}"))
                    elif len(ps) == 1 or (len(ps) == 2 and ps[1] != ps[0] + 1):
                        erros.append(("R4", f"{c}/{k}: bloco duplo inválido em {DIAS[dia]} {ps}"))
                elif len(ps) > 1:
                    erros.append(("R3", f"{c}/{k}: {len(ps)} aulas em {DIAS[dia]}"))

    # R5, R6
    prof_slot = defaultdict(int)
    for c, k, dia, p in L:
        prof_slot[(d["disc"][k]["prof"], dia, p)] += 1
    for (pr, dia, p), n in prof_slot.items():
        if n > 1:
            erros.append(("R5", f"{pr} com {n} aulas em {DIAS[dia]} T{p + 1}"))
        if (pr, dia, p) in d["indisp"]:
            erros.append(("R6", f"{pr} indisponível em {DIAS[dia]} T{p + 1}"))

    # R7
    pool_slot = defaultdict(int)
    for c, k, dia, p in L:
        pool_slot[(d["disc"][k]["pool"], dia, p)] += 1
    for (pool, dia, p), n in pool_slot.items():
        if n > d["salas"].get(pool, 0):
            erros.append(("R7", f"{pool}: {n} aulas > {d['salas'].get(pool, 0)} em {DIAS[dia]} T{p + 1}"))

    return erros


def testar_validador(d, aulas):
    """Corrompe um horário válido de modo a violar cada requisito e confirma que o validador deteta."""
    L = list(aulas)
    assert not validar(d, L), "o horário base devia ser válido"
    P = N_PERIODOS
    codigos = lambda x: {c for c, _ in validar(d, x)}
    res = {}

    c, k, dia, p = L[0]
    outro = next(kk for kk in d["disc"] if kk != k)
    res["R1"] = "R1" in codigos(L + [(c, outro, dia, p)])
    res["R2"] = "R2" in codigos(L[1:])

    nd = next((a for a in L if not d["disc"][a[1]]["duplo"]), None)
    if nd:
        res["R3"] = "R3" in codigos(L + [(nd[0], nd[1], nd[2], (nd[3] + 1) % P)])
    du = next((a for a in L if d["disc"][a[1]]["duplo"]), None)
    if du:
        res["R4"] = "R4" in codigos([a for a in L if a != du])

    if len(d["turmas"]) > 1:
        c2 = next(t for t in d["turmas"] if t != c)
        res["R5"] = "R5" in codigos(L + [(c2, k, dia, p)])

    for (pr, di, pi) in sorted(d["indisp"]):
        ks = [kk for kk, i in d["disc"].items() if i["prof"] == pr]
        if ks:
            res["R6"] = "R6" in codigos(L + [(d["turmas"][0], ks[0], di, pi)])
            break

    pool, qtd = next(iter(d["salas"].items()))
    kp = next(kk for kk, i in d["disc"].items() if i["pool"] == pool)
    res["R7"] = "R7" in codigos(L + [(f"fake{i}", kp, 0, 0) for i in range(qtd + 1)])

    return res

## 7. Horário inicial `H0` (dados/)

In [ ]:
d0 = carregar_dados("dados")
r0 = resolver(d0, tempo=30)
S0 = r0["S"]
aulas0 = blocos_para_aulas(d0, S0)

print("Estado:", r0["estado"], "| tempo:", round(r0["tempo"], 2), "s")
print("Buracos:", contar_buracos(d0, aulas0))
print("Erros de validação:", validar(d0, aulas0))
assert not validar(d0, aulas0)

for t in d0["turmas"]:
    print(f"\nHorário {t}")
    display(mostrar_horario(d0, S0, t))

In [ ]:
## Testes do validador (um caso por requisito)
resultado_testes = testar_validador(d0, aulas0)
print(resultado_testes)
assert all(resultado_testes.values())

## 8. Construção incremental (R9)

Estratégia em fases (vizinhanças crescentes, estilo LNS):

0. liberta só os pares `(turma, disciplina)` cujas aulas deixaram de ser válidas;
1. liberta também todas as disciplinas dos professores envolvidos;
2. liberta todas as disciplinas das turmas envolvidas;
3. resolve o modelo completo com `H0` como *hint*.

Em todas as fases o solver minimiza primeiro o nº de aulas alteradas e só depois os buracos. Pára na primeira fase que encontra solução.

In [ ]:
def afetados(d, S0v):
    """Pares (turma, disciplina) e professores cujas aulas de H0 deixaram de ser válidas com os dados d."""
    aulas = blocos_para_aulas(d, S0v)
    prof_slot, pool_slot = defaultdict(list), defaultdict(list)
    for (c, k, dia, p) in aulas:
        info = d["disc"][k]
        prof_slot[(info["prof"], dia, p)].append((c, k))
        pool_slot[(info["pool"], dia, p)].append((c, k))

    problemas, profs = set(), set()
    for (pr, dia, p), lst in prof_slot.items():
        if (pr, dia, p) in d["indisp"] or len(lst) > 1:       # R5/R6 violados
            problemas.update(lst)
            profs.add(pr)
    for (pool, dia, p), lst in pool_slot.items():
        if len(lst) > d["salas"].get(pool, 0):                # R7 violado
            problemas.update(lst)
            profs.update(d["disc"][k]["prof"] for _, k in lst)
    return problemas, profs


def reotimizar(d1, S0, tempo_fase=10.0):
    t0 = time.perf_counter()
    turmas_set = set(d1["turmas"])
    S0v = {key for key in S0 if key[0] in turmas_set and key[1] in d1["disc"]}

    problemas, profs = afetados(d1, S0v)
    pares = [(c, k) for c in d1["turmas"] for k in d1["disc"]]
    n0 = set(problemas)
    n1 = n0 | {(c, k) for (c, k) in pares if d1["disc"][k]["prof"] in profs}
    turmas_afet = {c for c, _ in n1}
    n2 = {(c, k) for (c, k) in pares if c in turmas_afet}

    for nome, livres in (("vizinhança 0 (aulas inválidas)", n0),
                         ("vizinhança 1 (+ professores)", n1),
                         ("vizinhança 2 (+ turmas)", n2)):
        r = resolver(d1, ref=S0v, livres=livres, tempo=tempo_fase)
        if r["S"] is not None:
            r["fase"] = nome
            r["tempo"] = time.perf_counter() - t0
            return r

    r = resolver(d1, ref=S0v, livres=None, tempo=tempo_fase * 3)   # modelo completo + hint
    r["fase"] = "global com hint de H0"
    r["tempo"] = time.perf_counter() - t0
    return r


def resumo(nome, d, r, S0):
    if r["S"] is None:
        return {"método": nome, "estado": r["estado"], "tempo (s)": round(r["tempo"], 2)}
    aulas = blocos_para_aulas(d, r["S"])
    return {
        "método": nome,
        "fase": r["fase"],
        "estado": r["estado"],
        "tempo (s)": round(r["tempo"], 2),
        "1ª solução (s)": None if r["t_primeira"] is None else round(r["t_primeira"], 2),
        "buracos": contar_buracos(d, aulas),
        "aulas alteradas vs H0": aulas_alteradas(S0, r["S"]),
        "válido (R1–R7)": not validar(d, aulas),
    }

In [ ]:
## Fluxo R9: dados/ -> dados_v2/
d1 = carregar_dados("dados_v2")

r_zero = resolver(d1, tempo=30)          # H1 do zero (sem usar H0)
r_inc = reotimizar(d1, S0)               # H1 incremental (usa H0)

tabela = pd.DataFrame([
    resumo("H1 do zero", d1, r_zero, S0),
    resumo("H1 incremental", d1, r_inc, S0),
])
display(tabela)
assert tabela["válido (R1–R7)"].all()

In [ ]:
for t in d1["turmas"]:
    print(f"\nHorário H1 (incremental) — {t}")
    display(mostrar_horario(d1, r_inc["S"], t))

## 9. Outros tipos de alteração de recursos

Funções que criam uma cópia dos dados com a alteração pedida (sem tocar nos ficheiros):
indisponibilidade/disponibilidade de professor, avaria de sala, turma nova e substituição de professor.
Cada cenário é resolvido de forma incremental e do zero, e ambos são validados.

In [ ]:
def alt_indisponibilidade(d, prof, dia, periodos, indisponivel=True):
    n = copy.deepcopy(d)
    for p in periodos:
        if indisponivel:
            n["indisp"].add((prof, DIAS.index(dia), p - 1))
        else:
            n["indisp"].discard((prof, DIAS.index(dia), p - 1))
    return n


def alt_sala(d, pool, nova_qtd):
    n = copy.deepcopy(d)
    n["salas"][pool] = nova_qtd
    return n


def alt_nova_turma(d, nome):
    n = copy.deepcopy(d)
    n["turmas"].append(nome)
    return n


def alt_substituir_prof(d, disciplina, novo_prof):
    n = copy.deepcopy(d)
    n["disc"][disciplina]["prof"] = novo_prof
    return n


# Cenários construídos a partir dos dados lidos (sem nomes escritos à mão)
ks = list(d0["disc"])
prof_a = d0["disc"][ks[0]]["prof"]
prof_b = d0["disc"][ks[1]]["prof"]

cenarios = {
    "Professor fica indisponível (Qua T1–T2)": alt_indisponibilidade(d0, prof_a, "Qua", [1, 2]),
    "Sala normal avariada (menos 1 sala)": alt_sala(d0, POOL_NORMAL, max(1, len(d0["turmas"]) - 1)),
    "Turma nova a meio do ano": alt_nova_turma(d0, "Turma_Nova"),
    "Professor substituído por outro já existente": alt_substituir_prof(d0, ks[0], prof_b),
}
if d0["indisp"]:
    pr, di, pi = sorted(d0["indisp"])[0]
    cenarios["Professor fica disponível em mais um tempo"] = alt_indisponibilidade(
        d0, pr, DIAS[di], [pi + 1], indisponivel=False)

linhas = []
for nome, dn in cenarios.items():
    r_i = reotimizar(dn, S0, tempo_fase=10)
    r_z = resolver(dn, tempo=15)
    for metodo, r in (("incremental", r_i), ("do zero", r_z)):
        linha = resumo(metodo, dn, r, S0)
        linha["cenário"] = nome
        linhas.append(linha)

display(pd.DataFrame(linhas).set_index(["cenário", "método"]))
# Nota: um cenário pode ser INFEASIBLE (ex.: demasiado restritivo); isso também é um resultado válido.

## 10. Teste com um conjunto de dados diferente (sem *hardcoding*)

Geramos em disco um dataset **diferente** (3 turmas, 7 disciplinas, outras exceções) e corremos exatamente o mesmo código.

In [ ]:
def escrever_dataset_teste(pasta):
    os.makedirs(pasta, exist_ok=True)
    pd.DataFrame({"turma": ["8ºA", "8ºB", "8ºC"]}).to_csv(f"{pasta}/turmas.csv", index=False)
    pd.DataFrame([
        ["Matemática", "Prof. Rui", 4, "nao", ""],
        ["Português", "Prof. Sara", 4, "nao", ""],
        ["Ciências", "Prof. Tiago", 3, "nao", "Laboratório"],
        ["Geografia", "Prof. Ugo", 2, "nao", ""],
        ["Inglês", "Prof. Ugo", 2, "nao", ""],
        ["Educação Física", "Prof. Vera", 2, "sim", "Ginásio"],
        ["Expressão Plástica", "Prof. Xavier", 2, "sim", ""],
    ], columns=["disciplina", "professor", "carga_semanal", "duplo_periodo", "sala_especial"]
    ).to_csv(f"{pasta}/disciplinas.csv", index=False)
    pd.DataFrame([
        ["Sala Normal", "normal", 6],
        ["Laboratório", "especial", 1],
        ["Ginásio", "especial", 1],
    ], columns=["sala", "tipo", "quantidade"]).to_csv(f"{pasta}/salas.csv", index=False)
    pd.DataFrame([
        ["Prof. Vera", "Seg", 1], ["Prof. Vera", "Seg", 2], ["Prof. Vera", "Ter", 1],
        ["Prof. Rui", "Sex", 4], ["Prof. Rui", "Sex", 5],
    ], columns=["professor", "dia", "periodo"]).to_csv(f"{pasta}/disponibilidade_excecoes.csv", index=False)


with tempfile.TemporaryDirectory() as tmp:
    escrever_dataset_teste(tmp)
    dt = carregar_dados(tmp)

rt = resolver(dt, tempo=30)
aulas_t = blocos_para_aulas(dt, rt["S"])
print("Estado:", rt["estado"], "| buracos:", contar_buracos(dt, aulas_t))
print("Erros de validação:", validar(dt, aulas_t))
assert not validar(dt, aulas_t)
print("Testes do validador:", testar_validador(dt, aulas_t))
display(mostrar_horario(dt, rt["S"], dt["turmas"][0]))

## 11. (Bónus) Escala

Geramos datasets sintéticos com mais turmas e medimos o tempo de resolução do zero e de uma reotimização incremental
(um professor fica indisponível a uma manhã). O formato dos CSV associa **um professor a cada disciplina**, por isso o
limite de escala é a capacidade semanal dos professores (25 tempos).

In [ ]:
def gerar_sintetico(pasta, n_turmas):
    os.makedirs(pasta, exist_ok=True)
    pd.DataFrame({"turma": [f"T{i + 1}" for i in range(n_turmas)]}).to_csv(f"{pasta}/turmas.csv", index=False)
    cargas = [3, 3, 2, 2, 2, 2, 2, 2]
    linhas = []
    for i, cg in enumerate(cargas):
        sala = "Laboratório" if i == 2 else ""
        dup = "sim" if i == 5 else "nao"
        if i == 5:
            sala = "Ginásio"
        linhas.append([f"Disc{i + 1}", f"Prof{i + 1}", cg, dup, sala])
    pd.DataFrame(linhas, columns=["disciplina", "professor", "carga_semanal", "duplo_periodo", "sala_especial"]
                 ).to_csv(f"{pasta}/disciplinas.csv", index=False)
    pd.DataFrame([
        ["Sala Normal", "normal", n_turmas],
        ["Laboratório", "especial", max(1, n_turmas // 2)],
        ["Ginásio", "especial", max(1, n_turmas // 3)],
    ], columns=["sala", "tipo", "quantidade"]).to_csv(f"{pasta}/salas.csv", index=False)
    pd.DataFrame([["Prof8", "Seg", 1]], columns=["professor", "dia", "periodo"]
                 ).to_csv(f"{pasta}/disponibilidade_excecoes.csv", index=False)


linhas = []
for n in [2, 4, 6]:
    with tempfile.TemporaryDirectory() as tmp:
        gerar_sintetico(tmp, n)
        ds = carregar_dados(tmp)
    r_a = resolver(ds, tempo=30)
    if r_a["S"] is None:
        linhas.append({"turmas": n, "estado H0": r_a["estado"]})
        continue
    dn = alt_indisponibilidade(ds, "Prof1", "Seg", [1, 2, 3])
    r_i = reotimizar(dn, r_a["S"], tempo_fase=10)
    r_z = resolver(dn, tempo=30)
    linhas.append({
        "turmas": n,
        "H0 (s)": round(r_a["tempo"], 2),
        "H1 do zero (s)": round(r_z["tempo"], 2),
        "H1 incremental (s)": round(r_i["tempo"], 2),
        "alteradas (incremental)": aulas_alteradas(r_a["S"], r_i["S"]) if r_i["S"] else None,
        "alteradas (do zero)": aulas_alteradas(r_a["S"], r_z["S"]) if r_z["S"] else None,
        "estado inc.": r_i["estado"],
    })
display(pd.DataFrame(linhas))

## 12. Conclusões e discussão

- O modelo cumpre R1–R7 (verificado por um validador independente do solver) e lê sempre os dados de ficheiros (R8).
- A construção incremental (R9) reaproveita `H0`: fixa o que não é afetado e resolve só o subproblema, expandindo a vizinhança se necessário. A tabela da secção 8 compara tempo e nº de aulas alteradas face a resolver do zero.
- Limites: o tempo do solver cresce com o nº de turmas; a escala está também limitada pela capacidade semanal dos professores (um professor por disciplina no formato dos CSV).
- Extensões possíveis: preferências dos professores (penalizar T1 ou T5) como termos adicionais na função objetivo.

### Como reproduzir
1. `pip install ortools pandas`
2. Colocar as pastas `dados/` e `dados_v2/` ao lado do notebook.
3. Executar todas as células por ordem.
4. Exportar para PDF: `jupyter nbconvert --to pdf horario_escolar.ipynb`